**Autor:** Alan Sastre  
**GitHub:** [github.com/alansastre/genai-agentes](https://github.com/alansastre/genai-agentes)

---

## La arquitectura de subagentes

Cuando un agente único acumula demasiadas herramientas y dominios, la calidad de sus decisiones se degrada. El modelo tiene dificultades para elegir correctamente entre veinte herramientas distintas, el historial de la conversación crece hasta saturar la ventana de contexto, y el resultado son respuestas menos precisas.

La arquitectura de **subagentes** aborda esto dividiendo la responsabilidad. Un agente principal, llamado *supervisor*, coordina a otros agentes especializados tratándolos como herramientas. Cada subagente opera en su propia ventana de contexto limpia, con solo las herramientas y conocimiento de su dominio.

> El beneficio clave no es solo la especialización: es el **aislamiento de contexto**. Cada subagente recibe exactamente la información que necesita, sin el ruido del historial completo de la conversación principal.

```mermaid
flowchart TB
    user([Usuario])
    sup[Supervisor]
    t1[tool: soporte_tecnico]
    t2[tool: soporte_comercial]
    sa1[Subagente tecnico]
    sa2[Subagente comercial]
    kb[Base de conocimiento]
    cat[Catalogo de precios]
    user --> sup
    sup --> t1
    sup --> t2
    t1 --> sa1
    t2 --> sa2
    sa1 --> kb
    sa2 --> cat
    sa1 --> t1
    sa2 --> t2
    t1 --> sup
    t2 --> sup
    sup --> user
```

Este patrón resulta adecuado cuando:

- El sistema maneja **dominios claramente diferenciados** (soporte técnico, facturación, logística)
- El historial de conversación crece tanto que empieza a afectar la calidad de las respuestas
- Distintos equipos desarrollan y mantienen agentes especializados de forma independiente

## Implementación del patrón supervisor

El mecanismo es el siguiente: se envuelve la invocación de cada subagente dentro de una función decorada con `@tool`. El supervisor la ve como cualquier otra herramienta y decide cuándo invocarla basándose en su nombre y descripción.

### Preparar el entorno

In [1]:
from langchain.agents import create_agent
from langchain.tools import tool
from langchain.messages import HumanMessage, AIMessage, ToolMessage

### Crear los subagentes especializados

Cada subagente se define con sus propias herramientas y su propio system prompt. El agente técnico no sabe nada de precios; el comercial, nada de especificaciones. Esta separación garantiza que cada uno responda exclusivamente dentro de su dominio:

In [2]:
@tool
def consultar_especificaciones(producto: str) -> str:
    """Consulta especificaciones tecnicas. Usa exactamente: router, camara o altavoz."""
    datos = {
        "router": "LED rojo: sin conexion WAN. Reinicio: mantener boton 10 s. Alcance: 120 m2.",
        "camara": "Resolucion: 4K a 30fps. Angulo: 130 grados. Almacenamiento SD: hasta 256 GB.",
        "altavoz": "Potencia: 20W RMS. Bluetooth 5.3. Autonomia: 12 horas.",
    }
    # Busqueda por coincidencia parcial para tolerar variaciones del LLM
    clave = next((k for k in datos if k in producto.lower()), None)
    return datos[clave] if clave else f"Producto '{producto}' no encontrado. Disponibles: router, camara, altavoz."

agente_tecnico = create_agent(
    model="gpt-5-mini",
    tools=[consultar_especificaciones],
    system_prompt=(
        "Eres un tecnico especialista en hardware. "
        "Para consultar datos de un producto llama a consultar_especificaciones "
        "con el nombre simple del producto: router, camara o altavoz. "
        "Responde de forma directa y concisa."
    ),
)

In [3]:
@tool
def consultar_precios(producto: str) -> str:
    """Consulta precios, planes y garantias. Usa exactamente: router, camara o altavoz."""
    catalogo = {
        "router": "Precio: 89 EUR. Garantia: 2 anios. Soporte 24/7 incluido.",
        "camara": "Precio: 149 EUR. Plan nube: 9 EUR/mes (30 dias de grabacion continua).",
        "altavoz": "Precio: 79 EUR. Garantia: 1 anio. Sin plan adicional.",
    }
    clave = next((k for k in catalogo if k in producto.lower()), None)
    return catalogo[clave] if clave else f"Producto '{producto}' no encontrado. Disponibles: router, camara, altavoz."

agente_comercial = create_agent(
    model="gpt-5-mini",
    tools=[consultar_precios],
    system_prompt=(
        "Eres un asesor comercial. "
        "Para consultar datos de un producto llama a consultar_precios "
        "con el nombre simple del producto: router, camara o altavoz. "
        "Responde de forma directa y concisa."
    ),
)

### Envolver los subagentes como herramientas

El paso siguiente es exponer cada subagente como herramienta. El **nombre y la descripción** son los datos que el supervisor usará para decidir a quién delegar cada consulta:

In [4]:
@tool("soporte_tecnico", description="Resuelve dudas tecnicas y problemas de configuracion de productos")
def invocar_tecnico(consulta: str) -> str:
    resultado = agente_tecnico.invoke({
        "messages": [{"role": "user", "content": consulta}]
    })
    return resultado["messages"][-1].content

@tool("soporte_comercial", description="Responde preguntas sobre precios, garantias y planes de contratacion")
def invocar_comercial(consulta: str) -> str:
    resultado = agente_comercial.invoke({
        "messages": [{"role": "user", "content": consulta}]
    })
    return resultado["messages"][-1].content

> La descripción de cada herramienta es el único mecanismo que tiene el supervisor para saber a quién delegar. Una descripción vaga o ambigua produce delegaciones incorrectas. Cuanto más específica y diferenciada sea respecto a las otras herramientas, mejor funciona el sistema.

### Crear el supervisor

El supervisor se crea con las dos herramientas-subagente. Su system prompt le indica que debe coordinar y delegar, no resolver directamente:

In [5]:
supervisor = create_agent(
    model="gpt-5-mini",
    tools=[invocar_tecnico, invocar_comercial],
    system_prompt=(
        "Eres un coordinador de atencion al cliente. "
        "Delega las consultas tecnicas a soporte_tecnico "
        "y las comerciales a soporte_comercial. "
        "Si una consulta mezcla ambos temas, invoca los dos especialistas."
    ),
)

## Ejemplo completo: atención al cliente multidominio

Con el sistema montado, se puede probar con una consulta que combine ambos dominios. El supervisor debería invocar a los dos subagentes y combinar sus respuestas en una sola respuesta al usuario:

In [6]:
respuesta = supervisor.invoke({
    "messages": [HumanMessage(content="Para la camara: cuanto cuesta y cuanta SD admite?")]
})

print(respuesta["messages"][-1].content)

He consultado a soporte_comercial y a soporte_tecnico. Resumen:

Soporte comercial
- Precio: 149 EUR (compra única).
- Opciones: compra única 149 EUR; plan nube 9 EUR/mes (incluye 30 días de grabación continua).
- Descuentos: promociones puntuales, descuentos por compra múltiple o planes anuales (varían según campaña/vendedor). Puedo comprobar ofertas activas si indicas modelo o vendedor.
- Garantía: legal mínima 2 años (UE). Puede haber garantías comerciales adicionales según distribuidor.

Soporte técnico
- Capacidad máxima de SD: hasta 256 GB (tarjetas SDXC). También admite SDHC y SD.
- Sistema de archivos: tarjetas >32 GB normalmente usan exFAT; SD/SDHC (≤32 GB) usan FAT32. Puedo confirmar si la cámara formatea en exFAT o permite FAT32 si me das el modelo.
- Recomendación de velocidad: para grabación 4K@30fps mínimo UHS-I U3 / V30 (Clase 10); recomendable UHS-I U3/V30 o superior (V60/V90) para mayor estabilidad.
- Buenas prácticas: formatear la tarjeta en la cámara antes de usarla;

### Inspeccionar el flujo de delegación

Para verificar que el supervisor efectivamente delegó en lugar de responder por su cuenta, se puede recorrer el historial de mensajes. Los mensajes de tipo `AIMessage` con `tool_calls` revelan qué subagente fue invocado y con qué argumento; los `ToolMessage` contienen la respuesta que devolvió cada subagente:

In [7]:
mensajes = respuesta["messages"]

for msg in mensajes:
    if isinstance(msg, AIMessage) and msg.tool_calls:
        for tc in msg.tool_calls:
            print(f"Subagente invocado: {tc['name']}")
            print(f"  Consulta delegada: {tc['args'].get('consulta', '')}")
    elif isinstance(msg, ToolMessage):
        print(f"Respuesta de {msg.name}: {msg.content[:120]}...")

Subagente invocado: soporte_comercial
  Consulta delegada: Consulta del cliente: 'Para la cámara: ¿cuánto cuesta?' Indica precio actual, opciones de compra/planes, posibles descuentos y condiciones de garantía. Si necesitas el modelo específico para dar precio exacto, solicítalo.
Subagente invocado: soporte_tecnico
  Consulta delegada: Consulta del cliente: 'Para la cámara: ¿cuánta SD admite?' Indica la capacidad máxima de tarjeta SD compatible (SD/SDHC/SDXC), formatos de archivo soportados (FAT32/exFAT), recomendaciones de velocidad/clase y si necesitas el modelo específico para confirmar, solicítalo.
Respuesta de soporte_comercial: Precio actual: 149 EUR.

Opciones/planes:
- Compra única: 149 EUR.
- Plan nube: 9 EUR/mes (incluye 30 días de grabación ...
Respuesta de soporte_tecnico: - Capacidad máxima: 256 GB — equivalente a tarjetas SDXC.  
- Tipos de tarjeta: admite SDXC (por tanto también SDHC/SD)....


La salida mostrará el nombre del subagente invocado, la consulta que recibió y el fragmento de respuesta que devolvió antes de que el supervisor redactara el mensaje final.